In [ ]:
!pip -q install umap-learn
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import numpy as np, pandas as pd
from pathlib import Path
from sklearn.dummy import DummyClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, accuracy_score, classification_report
import torch, numpy as np
from PIL import ImageOps
from datasets import load_dataset
from transformers import AutoModel, AutoProcessor
from sklearn.cluster import AgglomerativeClustering
from sklearn.model_selection import RepeatedStratifiedKFold, StratifiedGroupKFold
import time
from sklearn.model_selection import StratifiedKFold

In [ ]:
CACHE = Path("/content/drive/MyDrive/legal_idp/embeddings")

In [ ]:
SEED = 42
sp = pd.read_csv(CACHE / "split.csv")
assert (sp["id"].to_numpy() == np.load(CACHE / "ids.npy")).all()

y = sp["label"].to_numpy()
tr, va, te = [(sp["split"] == s).to_numpy() for s in ("train", "val", "test")]
models = {"siglip": "siglip_siglip-base-patch16-224.npy", "jina": "jina_v4.npy"}
summary = []

for name, file in models.items():
    X = np.load(CACHE / file); X = X / np.linalg.norm(X, axis=1, keepdims=True)
    macro = lambda m, idx: f1_score(y[idx], m.predict(X[idx]), average="macro")

    # baselines
    maj = DummyClassifier(strategy="most_frequent").fit(X[tr], y[tr])
    knn = KNeighborsClassifier(1, metric="cosine").fit(X[tr], y[tr])
    plain = LogisticRegression(C=1, max_iter=3000).fit(X[tr], y[tr])     # untuned LR = "before"

    # tune on VAL only
    val_scores = {C: macro(LogisticRegression(C=C, max_iter=3000, class_weight="balanced",
                                              random_state=SEED).fit(X[tr], y[tr]), va)
                  for C in [0.1, 1, 10, 100, 1000]}
    bestC = max(val_scores, key=val_scores.get)
    lr = LogisticRegression(C=bestC, max_iter=3000, class_weight="balanced", random_state=SEED).fit(X[tr], y[tr])

    # TEST is scored once, after every choice is made
    pred = lr.predict(X[te])
    summary.append(dict(model=name, best_C=bestC,
                        val_majority=macro(maj, va), val_1NN=macro(knn, va),
                        val_LR_untuned=macro(plain, va), val_LR_tuned=val_scores[bestC],
                        test_macroF1=macro(lr, te), test_acc=accuracy_score(y[te], pred)))

    print(f"\n{'='*20} {name} (C={bestC}) {'='*20}")
    print(classification_report(y[te], pred, digits=2, zero_division=0))
    wrong = sp[te].assign(true=y[te], pred=pred)
    wrong = wrong[wrong["true"] != wrong["pred"]][["id", "true", "pred", "document_quality"]]
    print(f"Misclassified: {len(wrong)} of {te.sum()} test docs"); print(wrong.to_string(index=False))
    print("\nTest accuracy by document_quality:")
    print(sp[te].assign(ok=(y[te] == pred)).groupby("document_quality")["ok"].agg(["mean", "count"]).round(3))

res = pd.DataFrame(summary).round(3)
res.to_csv(CACHE / "classification_results.csv", index=False)
res


==================== siglip (C=1000) ====================
                            precision    recall  f1-score   support

         ACCOUNT_STATEMENT       1.00      1.00      1.00         8
                BANK_CHECK       1.00      1.00      1.00         7
                     CHART       1.00      1.00      1.00         8
COMMERCIAL_LEASE_AGREEMENT       1.00      1.00      1.00         8
     CREDIT_CARD_STATEMENT       1.00      1.00      1.00         7
             DELIVERY_NOTE       1.00      1.00      1.00         8
      EQUIPMENT_INSPECTION       1.00      0.88      0.93         8
                 FORM_1040       1.00      1.00      1.00         7
                  GLOSSARY       0.89      1.00      0.94         8
                 NUTRITION       1.00      1.00      1.00         8
                    PATENT       1.00      1.00      1.00         8
            PATIENT_INTAKE       1.00      1.00      1.00         8
              PAY_IN_SHEET       1.00      1.00      1.0

,model,best_C,val_majority,val_1NN,val_LR_untuned,val_LR_tuned,test_macroF1,test_acc
0,siglip,1000,0.007,0.967,0.974,0.986,0.993,0.993
1,jina,1000,0.007,0.972,0.905,0.981,0.987,0.987


In [ ]:
X = np.load(CACHE / "siglip_siglip-base-patch16-224.npy"); X = X / np.linalg.norm(X, axis=1, keepdims=True)
lr = LogisticRegression(C=1000, max_iter=3000, class_weight="balanced", random_state=SEED).fit(X[tr], y[tr])
proba = lr.predict_proba(X[te])
out = sp[te][["id", "label"]].rename(columns={"label": "true"}).assign(
    predicted=lr.classes_[proba.argmax(1)], confidence=proba.max(1).round(3))
out["correct"] = out["true"] == out["predicted"]
out.sort_values("confidence").head(20)    # least confident first: where the model is least sure

,id,true,predicted,confidence,correct
902,902,EQUIPMENT_INSPECTION,EQUIPMENT_INSPECTION,0.196,True
453,453,EQUIPMENT_INSPECTION,GLOSSARY,0.292,False
399,399,DELIVERY_NOTE,DELIVERY_NOTE,0.369,True
716,716,NUTRITION,NUTRITION,0.438,True
548,548,CHART,CHART,0.581,True
858,858,Unknown,Unknown,0.816,True
724,724,Unknown,Unknown,0.829,True
694,694,Unknown,Unknown,0.845,True
22,22,CHART,CHART,0.872,True
638,638,Unknown,Unknown,0.875,True


In [ ]:
# classifier trained on the train split (same as before)
X = np.load(CACHE / "siglip_siglip-base-patch16-224.npy"); X = X / np.linalg.norm(X, axis=1, keepdims=True)
y = sp["label"].to_numpy(); tr = (sp["split"] == "train").to_numpy()
lr = LogisticRegression(C=1000, max_iter=3000, class_weight="balanced", random_state=42).fit(X[tr], y[tr])

ds = load_dataset("getomni-ai/ocr-benchmark", split="test").select_columns(["id", "image"])
mid = "google/siglip-base-patch16-224"
model = AutoModel.from_pretrained(mid).eval(); proc = AutoProcessor.from_pretrained(mid)

def emb(img):                      # same preprocessing as your embedding notebook
    img = img.convert("RGB"); s = max(img.size); img = ImageOps.pad(img, (s, s), color="white")
    with torch.inference_mode():
        f = model.get_image_features(pixel_values=proc(images=[img], return_tensors="pt")["pixel_values"])
        if not torch.is_tensor(f): f = f.pooler_output
        return torch.nn.functional.normalize(f.float(), dim=-1)[0].numpy()

for i in [453, 902]:
    for angle in [0, 90, 180, 270]:
        p = lr.predict_proba([emb(ds[i]["image"].rotate(angle, expand=True))])[0]
        print(f"doc {i} rotated {angle:3d}: {lr.classes_[p.argmax()]:22s} confidence {p.max():.3f}")

Resolving data files:   0%|          | 0/1001 [00:00<?, ?it/s]

config.json:   0%|          | 0.00/432 [00:00<?, ?B/s]

[transformers] Model config: bos_token_id must be `None` or an integer within the vocabulary (between 0 and 31999), got 49406. This may result in unexpected behavior.
[transformers] Model config: eos_token_id must be `None` or an integer within the vocabulary (between 0 and 31999), got 49407. This may result in unexpected behavior.


model.safetensors: reconstructing file:   0%|          |  0.00B /  813MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/408 [00:00<?, ?it/s]

... (3 outputs left out)

spiece.model: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/409 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.40M [00:00<?, ?B/s]

doc 453 rotated   0: GLOSSARY               confidence 0.291
doc 453 rotated  90: EQUIPMENT_INSPECTION   confidence 0.217
doc 453 rotated 180: GLOSSARY               confidence 0.248
doc 453 rotated 270: EQUIPMENT_INSPECTION   confidence 0.995
doc 902 rotated   0: EQUIPMENT_INSPECTION   confidence 0.196
doc 902 rotated  90: EQUIPMENT_INSPECTION   confidence 0.976
doc 902 rotated 180: EQUIPMENT_INSPECTION   confidence 0.999
doc 902 rotated 270: EQUIPMENT_INSPECTION   confidence 0.997


In [ ]:
SEED = 42
sp = pd.read_csv(CACHE / "split.csv"); y = sp["label"].to_numpy()
E = {}
for n, f in {"siglip": "siglip_siglip-base-patch16-224.npy", "jina": "jina_v4.npy"}.items():
    X = np.load(CACHE / f); E[n] = X / np.linalg.norm(X, axis=1, keepdims=True)

def cv_scores(splits):
    out = {n: [] for n in E}
    for tr, te in splits:
        for n, X in E.items():
            m = LogisticRegression(C=100, max_iter=3000, class_weight="balanced", random_state=SEED).fit(X[tr], y[tr])
            out[n].append(f1_score(y[te], m.predict(X[te]), average="macro"))
    return {n: np.array(v) for n, v in out.items()}

def report(title, r):
    d = r["siglip"] - r["jina"]
    print(f"\n{title} ({len(d)} folds)")
    for n, v in r.items(): print(f"  {n:7s} macro-F1 {v.mean():.3f} +/- {v.std():.3f}")
    print(f"  siglip - jina: {d.mean():+.3f} | siglip better {(d>0).sum()}/{len(d)}, jina better {(d<0).sum()}, ties {(d==0).sum()}")

# A: random stratified folds
A = list(RepeatedStratifiedKFold(n_splits=5, n_repeats=4, random_state=SEED).split(E["siglip"], y))
report("A. Random stratified 5-fold x4", cv_scores(A))

# B: template-aware. Groups = 5 sub-clusters per class, built on BOTH embeddings so neither model defines them
Z = np.hstack([E["siglip"], E["jina"]]) / np.sqrt(2)
groups = np.empty(len(y), dtype=object)
for c in np.unique(y):
    idx = np.where(y == c)[0]
    g = AgglomerativeClustering(5, linkage="ward").fit_predict(Z[idx])
    groups[idx] = [f"{c}_{i}" for i in g]
B = []
for rep in range(4):
    B += list(StratifiedGroupKFold(5, shuffle=True, random_state=SEED + rep).split(Z, y, groups))
report("B. Template-aware 5-fold x4", cv_scores(B))


A. Random stratified 5-fold x4 (20 folds)
  siglip  macro-F1 0.985 +/- 0.007
  jina    macro-F1 0.979 +/- 0.009
  siglip - jina: +0.006 | siglip better 14/20, jina better 6, ties 0

B. Template-aware 5-fold x4 (20 folds)
  siglip  macro-F1 0.825 +/- 0.103
  jina    macro-F1 0.785 +/- 0.105
  siglip - jina: +0.040 | siglip better 14/20, jina better 6, ties 0


In [ ]:
SEED = 42
sp = pd.read_csv(CACHE / "split.csv"); y = sp["label"].to_numpy(); N = len(y)
R = np.load(CACHE / "siglip_rot4.npy")                       # (4 angles, 1000 docs, 768)
R = R / np.linalg.norm(R, axis=2, keepdims=True)
ANGLES = [0, 90, 180, 270]
fit = lambda Xa, ya: LogisticRegression(C=100, max_iter=3000, class_weight="balanced", random_state=SEED).fit(Xa, ya)

# ---------- STEP 2: how many documents look rotated? ----------
# A classifier that never saw the document scores the TRUE class at each of the 4 rotations.
p_true = np.zeros((4, N))
for tr, te in StratifiedKFold(5, shuffle=True, random_state=SEED).split(R[0], y):
    m = fit(R[0][tr], y[tr]); col = {c: i for i, c in enumerate(m.classes_)}
    for a in range(4):
        P = m.predict_proba(R[a][te]); p_true[a, te] = P[np.arange(len(te)), [col[c] for c in y[te]]]
best = p_true.argmax(0)
print("True-class probability when the page is shown as stored (0 deg):")
print("  below 0.5:", int((p_true[0] < 0.5).sum()), "docs | below 0.9:", int((p_true[0] < 0.9).sum()), "docs")
for lo, hi in [(0.5, 0.8), (0.9, 0.9)]:
    fixed = (p_true[0] < lo) & (p_true.max(0) > hi)
    print(f"\nLooks rotated (true-class prob < {lo} as stored, but > {hi} after some rotation): {int(fixed.sum())} of {N} docs")
    d = sp[fixed].assign(best_angle=[ANGLES[b] for b in best[fixed]])
    print(d.groupby("best_angle").size().to_string()); print(d.groupby("split").size().to_string())
    print(d.groupby("label").size().sort_values(ascending=False).head(8).to_string())
print("\nDocs 453 and 902, true-class probability by angle:")
print(pd.DataFrame(p_true[:, [453, 902]].round(3), index=ANGLES, columns=[453, 902]))
looks_rotated = (p_true[0] < 0.5) & (p_true.max(0) > 0.8)
pd.DataFrame({"id": sp["id"], "label": y, "split": sp["split"], "looks_rotated": looks_rotated,
              "best_angle": [ANGLES[b] for b in best], **{f"p_true_{a}": p_true[i].round(3) for i, a in enumerate(ANGLES)}}
             ).to_csv(CACHE / "rotation_diagnosis.csv", index=False)

# ---------- STEP 3: does handling rotation improve macro-F1? ----------
# Every method uses the same folds. A rotated copy never leaves its original document's fold.
S = np.load(CACHE / "siglip_siglip-base-patch16-224.npy"); J = np.load(CACHE / "jina_v4.npy")
Z = np.hstack([S / np.linalg.norm(S, axis=1, keepdims=True), J / np.linalg.norm(J, axis=1, keepdims=True)]) / np.sqrt(2)
groups = np.empty(N, dtype=object)
for c in np.unique(y):
    idx = np.where(y == c)[0]; groups[idx] = [f"{c}_{i}" for i in AgglomerativeClustering(5, linkage="ward").fit_predict(Z[idx])]

def evaluate(splits):
    rows = []
    for tr, te in splits:
        base = fit(R[0][tr], y[tr])                                                   # 1. baseline: pages as stored
        aug = fit(np.concatenate([R[a][tr] for a in range(4)]), np.tile(y[tr], 4))    # 2. augmentation: 4 rotations of TRAIN docs
        P = np.stack([base.predict_proba(R[a][te]) for a in range(4)])                # 3. test-time: try 4 rotations
        best_a = P.max(2).argmax(0)                                                   #    keep the most confident one
        tta = base.classes_[P.argmax(2)[best_a, np.arange(len(te))]]
        f = lambda pred: f1_score(y[te], pred, average="macro")
        rows.append(dict(baseline=f(base.predict(R[0][te])), augmentation=f(aug.predict(R[0][te])), test_time_rotation=f(tta)))
    return pd.DataFrame(rows)

t = time.time()
A = list(RepeatedStratifiedKFold(n_splits=5, n_repeats=2, random_state=SEED).split(R[0], y))
B = [s for rep in range(2) for s in StratifiedGroupKFold(5, shuffle=True, random_state=SEED + rep).split(Z, y, groups)]
for title, splits in [("A. Random stratified 5-fold x2", A), ("B. Template-aware 5-fold x2", B)]:
    r = evaluate(splits); print(f"\n{title} ({len(r)} folds)")
    print(r.agg(["mean", "std"]).round(3).T.to_string())
    for m in ["augmentation", "test_time_rotation"]:
        d = r[m] - r["baseline"]; print(f"  {m} - baseline: {d.mean():+.3f} | better in {(d>0).sum()}/{len(d)}, worse in {(d<0).sum()}, ties {(d==0).sum()}")
print("\nruntime (s):", round(time.time() - t))

True-class probability when the page is shown as stored (0 deg):
  below 0.5: 27 docs | below 0.9: 113 docs

Looks rotated (true-class prob < 0.5 as stored, but > 0.8 after some rotation): 13 of 1000 docs
best_angle
90     5
180    5
270    3
split
test     3
train    9
val      1
label
DELIVERY_NOTE                 2
SHIPPING_INVOICE              2
EQUIPMENT_INSPECTION          2
PAY_IN_SHEET                  2
COMMERCIAL_LEASE_AGREEMENT    1
GLOSSARY                      1
CREDIT_CARD_STATEMENT         1
REAL_ESTATE                   1

Looks rotated (true-class prob < 0.9 as stored, but > 0.9 after some rotation): 43 of 1000 docs
best_angle
90     14
180    14
270    15
split
test      7
train    30
val       6
label
NUTRITION                     5
Unknown                       5
ACCOUNT_STATEMENT             3
CREDIT_CARD_STATEMENT         3
COMMERCIAL_LEASE_AGREEMENT    3
PAY_IN_SHEET                  3
REAL_ESTATE                   3
DELIVERY_NOTE                 2

Docs 453 and 